In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from data import *
from metrics import *
from scipy import ndimage

In [19]:
x, y = get_stanford_background()

In [ ]:
x = cv2.imread(x[0])
y = cv2.imread(y[0], cv2.IMREAD_GRAYSCALE)

In [ ]:
plt.imshow(y)

In [18]:
from ultralytics import FastSAM
model = FastSAM("./checkpoints/FastSAM-x.pt")

In [20]:
results = model(x[0], points=[[200, 200]], labels=[1])



image 1/1 /mnt/nvme0/home/utbt/KhoaVM/NoisySAM/data/stanford-bg/images/0000047.jpg: 768x1024 1 object, 257.8ms
Speed: 107.0ms preprocess, 257.8ms inference, 706.5ms postprocess per image at shape (1, 3, 768, 1024)


In [ ]:
def split_masks(y):
    masks = []

    ids = np.unique(y)

    for gid in ids:
        masks.append(y == gid)

    return masks

def sample_points(mask):
    labeled_mask, num_regions = ndimage.label(mask)
    points = []
    for region_id in range(1, num_regions + 1):
        region = labeled_mask == region_id
        dist = ndimage.distance_transform_edt(region)
        cy, cx = np.unravel_index(np.argmax(dist), dist.shape)
        points.append([cx, cy])

    return np.array(points)

def sam_predict_mask(predictor, image, points):

    labels = np.ones(len(points))

    masks, scores, _ = predictor.predict_inst(
        point_coords=points,
        point_labels=labels,
        multimask_output=False
    )

    return masks[0]

def evaluate_sam(image, gt_mask, predictor):

    masks = split_masks(gt_mask)

    ious = []
    dices = []
    precisions = []
    recalls = []

    preds = []

    predictor.set_image(image)

    for mask in masks:

        points = sample_points(mask)

        if points is None:
            continue

        pred = sam_predict_mask(predictor, image, points)
        preds.append(pred)

        metrics = compute_metrics(pred, mask)

        ious.append(metrics["iou"])
        dices.append(metrics["dice"])
        precisions.append(metrics["precision"])
        recalls.append(metrics["recall"])

    if len(ious) == 0:
        return None

    mean_metrics = {
        "miou": np.mean(ious),
        "mdice": np.mean(dices),
        "mprecision": np.mean(precisions),
        "mrecall": np.mean(recalls)
    }

    return mean_metrics, preds, masks


In [ ]:
metrics, preds, masks = evaluate_sam(x, y, processor)

In [ ]:
metrics

In [ ]:
n = len(masks)

plt.figure(figsize=(10, 4 * n))

for i in range(n):
    plt.subplot(n, 2, 2*i + 1)
    plt.imshow(preds[i])
    plt.axis("off")
    plt.title("pred")

    plt.subplot(n, 2, 2*i + 2)
    plt.imshow(masks[i])
    plt.axis("off")
    plt.title("mask")

plt.tight_layout()
plt.show()